# M2.5: Dual-Retriever Candidate Union

Reconstructed from the original M2.6 rho=0 baseline to make the accepted BPR top-500 + MiniLM metadata top-500 union reproducible. Retrieval only; reranking happens in M4 and M5. This is a reconstruction because the original M2.5 notebook backing file is not available.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path(".")

TRAIN_PATH = DATA_DIR / "train_interactions.parquet"
VALID_PATH = DATA_DIR / "valid_interactions.parquet"
TEST_PATH = DATA_DIR / "test_interactions.parquet"

BPR_CHECKPOINT = DATA_DIR / "bpr_m1a_all_interactions.pt"
M2_EMBED_PATH = DATA_DIR / "metadata_minilm_embeddings.npy"
M2_INDEX_PATH = DATA_DIR / "metadata_embedding_index.parquet"

CANDIDATE_K = 500
BATCH_SIZE = 128
RECENT_N = 2
RHO_VALUES = [0.0]

print("Candidate K:", CANDIDATE_K)
print("Batch size :", BATCH_SIZE)
print("Recent N   :", RECENT_N)
print("Rho values :", RHO_VALUES)

## 2. Load interaction splits and saved M1/M2 representations

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
valid = pd.read_parquet(VALID_PATH)
test = pd.read_parquet(TEST_PATH)

metadata_embeddings = np.load(M2_EMBED_PATH).astype(np.float32)
metadata_index = pd.read_parquet(M2_INDEX_PATH)

# Normalize metadata vectors again for safety.
meta_norms = np.linalg.norm(metadata_embeddings, axis=1, keepdims=True)
metadata_embeddings = metadata_embeddings / np.clip(meta_norms, 1e-12, None)

state_dict = torch.load(BPR_CHECKPOINT, map_location="cpu")
bpr_user_embeddings = (
    state_dict["user_embedding.weight"]
    .cpu().numpy().astype(np.float32)
)
bpr_item_embeddings = (
    state_dict["item_embedding.weight"]
    .cpu().numpy().astype(np.float32)
)

print("Train:", train.shape)
print("Valid:", valid.shape)
print("Test :", test.shape)
print("Metadata embeddings:", metadata_embeddings.shape)
print("BPR user embeddings:", bpr_user_embeddings.shape)
print("BPR item embeddings:", bpr_item_embeddings.shape)

## 3. Reconstruct the exact M1a mappings

The saved BPR checkpoint used first-appearance ordering:

- users: `TRAIN`
- items: `TRAIN -> VALID -> TEST`

Do not sort these IDs.

In [ ]:
all_users = train["user_id"].unique()

all_items = pd.concat([
    train["parent_asin"],
    valid["parent_asin"],
    test["parent_asin"],
]).unique()

user_to_idx = {user: idx for idx, user in enumerate(all_users)}
item_to_idx = {item: idx for idx, item in enumerate(all_items)}

NUM_USERS = len(all_users)
NUM_ITEMS = len(all_items)

assert NUM_USERS == bpr_user_embeddings.shape[0]
assert NUM_ITEMS == bpr_item_embeddings.shape[0]

print("Users:", NUM_USERS)
print("Items:", NUM_ITEMS)
print("BPR mapping aligned.")

## 4. Build TRAIN indices, seen-item masks, and retrieval catalogs

In [ ]:
train_indexed = train[["user_id", "parent_asin", "timestamp"]].copy()
train_indexed["user_idx"] = (
    train_indexed["user_id"].map(user_to_idx).astype(np.int32)
)
train_indexed["item_idx"] = (
    train_indexed["parent_asin"].map(item_to_idx).astype(np.int32)
)

user_seen = (
    train_indexed
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

# BPR can safely retrieve only items whose embeddings were trained in TRAIN.
bpr_catalog_idx = np.array(
    sorted({item_to_idx[item] for item in train["parent_asin"].unique()}),
    dtype=np.int32,
)

global_to_bpr_pos = np.full(NUM_ITEMS, -1, dtype=np.int32)
global_to_bpr_pos[bpr_catalog_idx] = np.arange(
    len(bpr_catalog_idx), dtype=np.int32
)

print("BPR catalog size:", len(bpr_catalog_idx))

## 5. Align metadata embeddings to the global item mapping

In [ ]:
asin_to_meta_idx = {
    asin: idx
    for idx, asin in enumerate(metadata_index["parent_asin"])
}

global_to_meta_idx = np.array(
    [asin_to_meta_idx.get(asin, -1) for asin in all_items],
    dtype=np.int32,
)

missing_metadata = int((global_to_meta_idx < 0).sum())
print("Global items missing metadata:", missing_metadata)
assert missing_metadata == 0

# Metadata vectors in the same global item order as BPR.
global_metadata_embeddings = metadata_embeddings[global_to_meta_idx]
metadata_catalog_embeddings = global_metadata_embeddings

train_indexed["meta_idx"] = (
    train_indexed["parent_asin"]
    .map(asin_to_meta_idx)
    .astype(np.int32)
)

print("Metadata catalog size:", NUM_ITEMS)

## 6. Build all-history and recent-2 metadata profiles

The baseline profile aggregates every TRAIN interaction for a user.

The recency profile uses only the user's **latest two TRAIN interactions by timestamp**.

Both components are L2-normalized before blending so that one branch does not dominate merely because it contains more interactions.

In [ ]:
def l2_normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    ok = norms[:, 0] > 1e-12
    matrix[ok] /= norms[ok]
    return matrix, ok


# ------------------------------------------------------------
# A. All-history profile
# ------------------------------------------------------------
all_history_matrix = csr_matrix(
    (
        np.ones(len(train_indexed), dtype=np.float32),
        (train_indexed["user_idx"], train_indexed["meta_idx"]),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

all_history_profiles = np.asarray(
    all_history_matrix @ metadata_embeddings,
    dtype=np.float32,
)
all_history_profiles, has_all_profile = l2_normalize_rows(
    all_history_profiles
)


# ------------------------------------------------------------
# B. Latest RECENT_N interactions per user
# ------------------------------------------------------------
chronological = train_indexed.sort_values(
    ["user_idx", "timestamp"],
    kind="mergesort",
)

recent_rows = (
    chronological
    .groupby("user_idx", sort=False, group_keys=False)
    .tail(RECENT_N)
)

recent_matrix = csr_matrix(
    (
        np.ones(len(recent_rows), dtype=np.float32),
        (recent_rows["user_idx"], recent_rows["meta_idx"]),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

recent_profiles = np.asarray(
    recent_matrix @ metadata_embeddings,
    dtype=np.float32,
)
recent_profiles, has_recent_profile = l2_normalize_rows(
    recent_profiles
)

print("All-history profiles:", all_history_profiles.shape)
print("Recent profiles     :", recent_profiles.shape)
print("Users with all profile   :", int(has_all_profile.sum()))
print("Users with recent profile:", int(has_recent_profile.sum()))
print("Recent interactions used :", len(recent_rows))

## 7. Prepare VALID users and targets

In [ ]:
valid_user_idx = np.array(
    [user_to_idx[user] for user in valid["user_id"].to_numpy()],
    dtype=np.int32,
)

valid_target_idx = np.array(
    [item_to_idx[item] for item in valid["parent_asin"].to_numpy()],
    dtype=np.int32,
)

NUM_VALID = len(valid_user_idx)

print("Validation users:", NUM_VALID)

## 8. Evaluate recency-aware metadata retrieval

For each validation batch we compute only three expensive score matrices:

1. BPR scores
2. all-history metadata scores
3. recent-2 metadata scores

For each `rho`, the recency-aware metadata score is then obtained cheaply as:

`(1 - rho) * all_history_score + rho * recent_score`

Normalizing the blended user vector is unnecessary for ranking because it would multiply every item score for that user by the same positive scalar.

In [ ]:
def topk_sorted(scores, k):
    """Return sorted top-k column positions and their scores."""
    local = np.argpartition(scores, -k, axis=1)[:, -k:]
    top_scores = np.take_along_axis(scores, local, axis=1)
    order = np.argsort(top_scores, axis=1)[:, ::-1]
    local = np.take_along_axis(local, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)
    return local, top_scores


def update_top10_metrics(candidates, targets):
    top10 = candidates[:, :10]
    matches = top10 == targets[:, None]
    hits = matches.any(axis=1)

    hit_count = int(hits.sum())
    mrr = 0.0
    ndcg = 0.0

    if hits.any():
        ranks = np.argmax(matches[hits], axis=1) + 1
        mrr = float((1.0 / ranks).sum())
        ndcg = float((1.0 / np.log2(ranks + 1)).sum())

    return hit_count, mrr, ndcg


stats = {
    rho: {
        "meta_hits_500": 0,
        "union_hits_500": 0,
        "meta_only_hits": 0,
        "both_hits": 0,
        "meta_hr10": 0,
        "meta_mrr10": 0.0,
        "meta_ndcg10": 0.0,
    }
    for rho in RHO_VALUES
}

bpr_hits_500 = 0
bpr_hr10 = 0
bpr_mrr10 = 0.0
bpr_ndcg10 = 0.0

bpr_catalog_embeddings = bpr_item_embeddings[bpr_catalog_idx]

for start in tqdm(
    range(0, NUM_VALID, BATCH_SIZE),
    desc="Recency metadata evaluation",
):
    end = min(start + BATCH_SIZE, NUM_VALID)
    batch_users = valid_user_idx[start:end]
    batch_targets = valid_target_idx[start:end]

    # ------------------------------------------------------------
    # A. BPR top-500, unchanged from M2.5
    # ------------------------------------------------------------
    bpr_scores = (
        bpr_user_embeddings[batch_users]
        @ bpr_catalog_embeddings.T
    )

    for row, user_idx in enumerate(batch_users):
        seen = user_seen.get(int(user_idx), set())
        if not seen:
            continue

        seen_global = np.fromiter(seen, dtype=np.int32)
        seen_pos = global_to_bpr_pos[seen_global]
        seen_pos = seen_pos[seen_pos >= 0]
        bpr_scores[row, seen_pos] = -np.inf

    bpr_local, _ = topk_sorted(bpr_scores, CANDIDATE_K)
    bpr_candidates = bpr_catalog_idx[bpr_local]

    bpr_hit = (bpr_candidates == batch_targets[:, None]).any(axis=1)
    bpr_hits_500 += int(bpr_hit.sum())

    h, rr, dcg = update_top10_metrics(bpr_candidates, batch_targets)
    bpr_hr10 += h
    bpr_mrr10 += rr
    bpr_ndcg10 += dcg

    # ------------------------------------------------------------
    # B. Metadata component score matrices
    # ------------------------------------------------------------
    all_scores = (
        all_history_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )

    recent_scores = (
        recent_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )

    # Mask TRAIN-seen items from both components before blending.
    for row, user_idx in enumerate(batch_users):
        seen = user_seen.get(int(user_idx), set())
        if seen:
            seen_global = np.fromiter(seen, dtype=np.int32)
            all_scores[row, seen_global] = -np.inf
            recent_scores[row, seen_global] = -np.inf

    # ------------------------------------------------------------
    # C. Test each rho without another expensive embedding matmul
    # ------------------------------------------------------------
    for rho in RHO_VALUES:
        if rho == 0.0:
            meta_scores = all_scores
        else:
            # Seen entries are -inf in both matrices, so remain masked.
            meta_scores = (
                (1.0 - rho) * all_scores
                + rho * recent_scores
            )

        meta_local, _ = topk_sorted(meta_scores, CANDIDATE_K)
        meta_candidates = meta_local.astype(np.int32)

        meta_hit = (
            meta_candidates == batch_targets[:, None]
        ).any(axis=1)
        union_hit = bpr_hit | meta_hit

        stats[rho]["meta_hits_500"] += int(meta_hit.sum())
        stats[rho]["union_hits_500"] += int(union_hit.sum())
        stats[rho]["meta_only_hits"] += int((meta_hit & ~bpr_hit).sum())
        stats[rho]["both_hits"] += int((meta_hit & bpr_hit).sum())

        h, rr, dcg = update_top10_metrics(
            meta_candidates, batch_targets
        )
        stats[rho]["meta_hr10"] += h
        stats[rho]["meta_mrr10"] += rr
        stats[rho]["meta_ndcg10"] += dcg

## 9. Compare baseline and recency-aware profiles

In [ ]:
bpr_recall_500 = bpr_hits_500 / NUM_VALID
baseline_union = stats[0.0]["union_hits_500"] / NUM_VALID

rows = []
for rho in RHO_VALUES:
    s = stats[rho]
    meta_recall = s["meta_hits_500"] / NUM_VALID
    union_recall = s["union_hits_500"] / NUM_VALID

    rows.append({
        "Profile": (
            "All-history"
            if rho == 0.0
            else f"All + recent-{RECENT_N}"
        ),
        "rho": rho,
        "Metadata Recall@500": meta_recall,
        "Union Recall@500": union_recall,
        "Union gain vs baseline": union_recall - baseline_union,
        "Metadata-rescued targets": s["meta_only_hits"],
        "Metadata HR@10": s["meta_hr10"] / NUM_VALID,
        "Metadata MRR@10": s["meta_mrr10"] / NUM_VALID,
        "Metadata NDCG@10": s["meta_ndcg10"] / NUM_VALID,
    })

results = pd.DataFrame(rows)

print("BPR Recall@500:", bpr_recall_500)
print("BPR HR@10:", bpr_hr10 / NUM_VALID)
print("BPR MRR@10:", bpr_mrr10 / NUM_VALID)
print("BPR NDCG@10:", bpr_ndcg10 / NUM_VALID)
print()

results